In [1]:
# env/quadcopter_hover.py
import numpy as np
import gymnasium as gym

from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ObservationType, ActionType

class QuadcopterHoverEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"]}

    def __init__(self, render_mode=None, record_csv=False):
        super().__init__()
        self.render_mode = render_mode
        self.record_csv = record_csv

        self.env = HoverAviary(
            gui=(render_mode == "human"),
            obs=ObservationType("kin"),
            act=ActionType("one_d_rpm"),   # later: "rpm" or "pid"
            record=False,
        )

        self.action_space = self.env.action_space
        self.observation_space = self.env.observation_space

    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options or {})
        obs = np.asarray(obs, dtype=np.float32)
        return obs, info

    def step(self, action):
        action = np.asarray(action, dtype=np.float32)

        if hasattr(self.action_space, "low") and hasattr(self.action_space, "high"):
            action = np.clip(action, self.action_space.low, self.action_space.high)

        obs, reward, terminated, truncated, info = self.env.step(action)
        obs = np.asarray(obs, dtype=np.float32)

        return obs, float(reward), terminated, truncated, info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()

C:\GitHub\gym-pybullet-drones\gym_pybullet_drones\envs\BaseAviary.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from gym_pybullet_drones.envs.HoverAviary import HoverAviary

env = HoverAviary()
obs, _ = env.reset()
print(obs.shape)

[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 3.160000e-10, km 7.940000e-12,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
(1, 72)


c:\Users\sofie\anaconda3\envs\thesis-1\lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\sofie\anaconda3\envs\thesis-1\lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [3]:
# env = HoverAviary(gui=True)

In [4]:
from stable_baselines3 import PPO
from gym_pybullet_drones.envs.HoverAviary import HoverAviary

env = HoverAviary(gui=True)
model = PPO("MlpPolicy", env, verbose=1)
model.learn(10000)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 3.160000e-10, km 7.940000e-12,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
viewMatrix (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
projectionMatrix (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)


c:\Users\sofie\anaconda3\envs\thesis-1\lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\sofie\anaconda3\envs\thesis-1\lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
c:\Users\sofie\anaconda3\envs\thesis-1\lib\site-packages\torch\cuda\__init__.py:184: UserWarning: cudaGetDeviceCount() returned cudaErrorNotSupported, likely using older driver or on CPU machine (Triggered internally at C:\bld\libtorch_1770197068560\work\c10\cuda\CUDAFunctions.cpp:88.)
  return torch._C._cuda_getDeviceCount() > 0


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 12.5     |
|    ep_rew_mean     | 16.3     |
| time/              |          |
|    fps             | 40       |
|    iterations      | 1        |
|    time_elapsed    | 50       |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 13.7        |
|    ep_rew_mean          | 17.3        |
| time/                   |             |
|    fps                  | 38          |
|    iterations           | 2           |
|    time_elapsed         | 105         |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.017731896 |
|    clip_fraction        | 0.134       |
|    clip_range           | 0.2         |
|    entropy_loss   

KeyboardInterrupt: 